# Phase 7 — Calibration transfer có đối chứng, vòng 2

**Notebook mới, chưa chạy GPU.** Chạy ba code cell theo thứ tự trên runtime Colab mới.

Giữ nguyên P0 và incumbent. Với mỗi checkpoint, so **calibration CFE đã có** với **calibration fit chỉ trên clean `val_calib` v2**. Chấm hai operating point trên cùng logits `val_select` clean/Q30. Không train, không chấm CFE evaluation, không thay model `24869052…`.

Source Git nhánh **`hoang`**, khóa commit đã kiểm thử **`253942a5262432fbed7ce7ca8b12b4e455bfc2ce`**. Không cần ZIP code. 30 test CPU/code đã đạt; chúng không phải bằng chứng chạy GPU thật.

- `val_calib`: 1.080 ảnh, 590 real/490 fake; không trùng IDs với `val_select`.
- Fit T và threshold trên clean calibration; một threshold áp dụng cả clean/Q30.
- FPR≤5% trên 590 real cho phép tối đa **29 false positives**, comparator `>=` theo float32.
- Chỉ reuse validation logits khi source inference/GPU/Torch/CUDA/batch khớp. Nếu khác, runner đo lại validation cho cả hai model. GPU không bắt buộc T4.
- Calibration mới là artifact nghiên cứu `deployable=false`, không tự xuất vào package.

## Input Drive cần giữ nguyên

```text
MyDrive/Fatformer/
  phase7/train_crossdomain_v2.tar
  phase7/improvement_round1/pilot_20261109_182359/p0_legacy_full/best.pth
  checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth
  phase7/improvement_round2/diagnostics/val_diag_20261110_142828/
    ...16 file kết quả diagnostics đã hậu kiểm...
```

Nếu reference folder chỉ còn `validation_diagnostics_without_weights.zip`, runner đọc đúng các file đã khóa từ ZIP này. Hai calibration CFE control đi kèm source Git, đúng checkpoint và hash; không cần lấy lại ảnh CFE. Không sửa notebook diagnostics đã chạy.

## Cell 1 — Kiểm tra GPU

Chọn **Runtime → Change runtime type → GPU**, dùng runtime mới. Probe cả hai model ở Cell 3 mới xác nhận batch phù hợp.

In [ ]:
import torch

assert torch.cuda.is_available(), 'Chọn Runtime → Change runtime type → GPU trước khi chạy'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f'VRAM trống/tổng: {free_bytes / 1024**3:.2f}/{total_bytes / 1024**3:.2f} GiB')

## Cell 2 — Clone `hoang` và khóa source

Không ghi đè repo đã tồn tại. Không `pull` hoặc đổi commit giữa một run.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/content/fatformer-xla')
PINNED_COMMIT = '253942a5262432fbed7ce7ca8b12b4e455bfc2ce'
assert not REPO.exists(), 'Repo đã tồn tại: giữ log cũ và dùng runtime mới.'

subprocess.run([
    'git', 'clone', '--branch', 'hoang', '--single-branch',
    'https://github.com/LeeVHoangtk3/fatformer-xla.git', str(REPO)
], check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
subprocess.run([
    'git', 'merge-base', '--is-ancestor', PINNED_COMMIT, 'origin/hoang'
], cwd=REPO, check=True)
TRANSFER_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()
assert TRANSFER_COMMIT == PINNED_COMMIT
assert (REPO / 'notebooks/task/phase7_calibration_transfer/start_colab.py').is_file()
print('Source đã khóa:', TRANSFER_COMMIT)

## Cell 3 — Fit calibration độc lập và so sánh

Mount Drive khi Colab yêu cầu. Runner hash/extract archive lên SSD, kiểm source reference/weights/ảnh, probe và chốt batch chung. Việc đọc/hash archive hoặc copy weights có thể mất vài phút.

T được fit bằng solver NLL có kiểm tra hội tụ, trên `val_calib`. `val_select` chỉ dùng chấm điểm. Hai point luôn dùng cùng validation logits; nếu phải đo lại, cả hai model đều được đo lại cùng protocol.

Nếu OOM/lỗi: giữ notebook và run đã ghi, dùng runtime/run ID mới. Không tự giảm batch giữa run. ZIP tải xuống chỉ là ZIP kết quả, không chứa weights.

In [ ]:
TRANSFER_REPO = str(REPO)
TRANSFER_MAX_BATCH = 16

exec(compile(
    (REPO / 'notebooks/task/phase7_calibration_transfer/start_colab.py').read_text(encoding='utf-8'),
    'start_colab.py', 'exec'
))

## Sau khi chạy xong

```text
MyDrive/Fatformer/phase7/improvement_round2/calibration_transfer/<TRANSFER_RUN_ID>/
  identity.json
  batch_contract.json
  status.json
  candidate_p0/comparison/summary.csv
  candidate_p0/comparison/paired_deltas.csv
  candidate_p0/comparison/v2_calibration.json
  incumbent/comparison/summary.csv
  incumbent/comparison/paired_deltas.csv
  incumbent/comparison/v2_calibration.json
  report.json
  calibration_transfer_without_weights.zip
```

Đọc `summary.csv` từng model: so `cfe_control` và `v2_clean_fit` trên cả clean/Q30, xem recall Real/Fake và FPR cùng nhau. `paired_deltas.csv` là V2 minus control, CI theo ảnh conditional on fit. `group_metrics.csv`/`errors.csv` cho biết tradeoff từng nhóm và ảnh lỗi.

Lưu notebook thực thi vào:

```text
notebooks/task/phase7_calibration_transfer/phase7_calibration_transfer_executed.ipynb
```

Đưa kết quả giải nén vào:

```text
wait/calibration_transfer_without_weights/
```

Sau hậu kiểm mới quyết định bước robustness tiếp; không chạy lại 7.14/7.15 hoặc thay package ở đây.

## Phạm vi kết luận

- `val_select` đã dùng chọn P0 và phân tích lỗi; không phải test mù mới.
- Recipe này cùng đổi domain, cỡ mẫu và tỷ lệ lớp calibration; chưa tách nguyên nhân riêng từng yếu tố.
- Các điều kiện cache được kiểm là source inference/GPU/Torch/CUDA/batch đã ghi nhận; không coi đó là chứng minh mọi thư viện hay toàn bộ môi trường lịch sử giống nhau.
- Calibration không làm raw logits xếp hạng tốt hơn. Report dùng raw-margin AUC riêng; score AUC có thể đổi vì ties/bão hòa float32.
- ECE là confidence ECE 10 bins với argmax 0,5; không phải độ tin cậy của nhãn tại threshold vận hành.
- CI không bao quát fit uncertainty, phụ thuộc generator hoặc nhiều seed train.
- Guard Real ≥90% là chẩn đoán; không tự đóng G7.2/G7.3. Model báo cáo giữ nguyên tới khi có bằng chứng độc lập phù hợp.